## Import everything we need

In [46]:
import os
from dotenv import load_dotenv

#Langchain
from langchain_community.document_loaders import TextLoader
#To split the document into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter
# To embed the document chunks
from langchain_community.embeddings import JinaEmbeddings


In [8]:
load_dotenv()

True

In [9]:
gorq_key = os.getenv("GORQ_API_KEY")
jina_key = os.getenv("JINA_API_KEY")

print("Environmental variables loaded successfully.")

Environmental variables loaded successfully.


### Loading Data

In [10]:
DATA_FILE_PATH = os.path.join("data", "hr_text.txt")

Data Ingestion

In [13]:
loader = TextLoader(DATA_FILE_PATH,encoding="utf-8")

documents = loader.load()

print("Data Loaded successfully.")
print("="*50)
print(documents)


Data Loaded successfully.
[Document(metadata={'source': 'data\\hr_text.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)\n\n1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.\n\n2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to 4 PM.\n\n3. PROBATION PERIOD\nAll new employees undergo a probation period of 3 months from their date of 

### Lanchain Documents

Langchain processes everything in form of documents

Documents:
Page Contents -- the actual data
Metadata--extra information about the data

In [14]:
len(documents)

1

In [18]:
documents[0].metadata

{'source': 'data\\hr_text.txt'}

In [19]:
print("Total characters in the document:", len(documents[0].page_content))

Total characters in the document: 2597


### Splitting our data

In [36]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)

chunks = text_splitter.split_documents(documents)

print(len(chunks))

9


Now each splited chunk is a documnet i.e. each chunk page contains and metadata

In [37]:
print(chunks[0])

page_content='COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)' metadata={'source': 'data\\hr_text.txt'}


In [40]:
print(chunks[7].page_content)

7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.


### Embedding our data

In [47]:
embeddings_model = JinaEmbeddings(model_name="jina-embeddings-v2-base-en")

print("Embedding model name is ", embeddings_model.model_name)

Embedding model name is  jina-embeddings-v2-base-en


### Store Data in Vector DB

In [49]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(chunks, embeddings_model)

print("Vector store created successfully ", vector_store.index.ntotal)


Vector store created successfully  9


In [51]:
test_query = "How many sick leaves employees get"

##similarity search

top_matches = vector_store.similarity_search(test_query,k=2)

print(f"Query : {test_query}")
for i, match in enumerate(top_matches):
    print(f"---- Match {i+1} ----")
    print(match.page_content)
    print("="*50)

Query : How many sick leaves employees get
---- Match 1 ----
1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.
---- Match 2 ----
7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.


#### Tool

In [76]:
# Top 3 relevent documents for the query
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 3})
def search_hr_policy(question:str)-> str:
    """"
    Search the  HR policy document for the given question and return the answer.
    """
    matching_chunks = retriever.invoke(question)
    return "\n\n.join([chunk.page_content for chunk in matching_chunks])"

#### Data Retrieval

LLM

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.2
)

llm.model_name

'openai/gpt-oss-120b'

In [63]:
test_response = llm.invoke("Hey is learing Rag hard? reply in joke manner, in one line")

In [65]:
print(test_response.content)

Learning RAG isn’t hard—just convince your brain to fetch jokes faster than your search engine!


AI Agents can do 3 things

LLM --- Brain

Tool --- Superpower (here ingesting our own data)

Memory --- here we aren't using

In [66]:
from langchain.agents import create_agent


In [ ]:

hr_assistant = create_agent(
    model=llm,
    tools=[search_hr_policy],
    system_prompt ="""
    You are a friendly HR assistant. 
    Always use the search_hr_policy_tool to look up the fact before answering.
    If the answer is not there in search results, you should say "I don't know"
    instead of guessing.
    """ 
)

print("HR assistant is ready to answer")

HR assistant is ready answer


In [ ]:
def ask_hr_assistant(question: str) -> str:
    """Send a question to the RAG agent and print a nicely fromatted answer"""
    print("="*50)
    print(f"Question: {question}")
    print("-"*50)
    response = hr_assistant.invoke({"messages": [{"role": "user", "content": question}]})
    answer = response["messages"][-1].content
    
    print(f"Answer: {answer}")
    print("="*50)
    print()
    return answer

In [99]:
response = hr_assistant.invoke(
    {
        "messages": [
            {"role": "user", 
             "content": "Tell me about the sick leave policy"
            }
        ]
    }
)

In [100]:
print(response['messages'][-1].content)

I don't know.
